# 4 · How realistic is this market?

Realism here is a set of measurements against real markets, with what the
model still gets wrong written down beside them. There is no single score.
Two checks describe the shipped default:

- **The one-year panel.** Statistics of daily returns, volume and
  correlation, the index's yearly return and how the VIX reacts to a bad
  day, each measured on thirty seeds of one simulated year and compared
  with the range real markets have produced.
- **The long-run check.** Ninety simulated histories of twenty-one years
  each, plus replays of 2008 and 2020, the real 2020-21 and 2022 macro
  paths and the packaged recession, read against forty criteria a user
  would notice: how violent crashes are, how often fear takes hold, how
  many bear markets a decade brings, what the index returns over the long
  run, whether a rule that reads only prices, published fundamentals or
  published macro data finds an edge real markets do not offer, what size
  costs in the book, how bonds move beside stocks, and how a market
  recovers.

The first cell confirms that the preset you are running is the one the
tables below describe.

In [1]:
import math

import tradefloor as tf
from tradefloor import envelope as env
from tradefloor import facts
from tradefloor.facts import LABELS

running = tf.model_preset()["name"]
print("preset you are running      :", running)
print("preset the envelope describes:", env.PRESET)
print("certified horizon           :", env.CERTIFIED_HORIZON_DAYS, "trading days")
assert running == env.PRESET, "the tables below describe another preset"

preset you are running      : pt-v20
preset the envelope describes: pt-v20
certified horizon           : 252 trading days


## The one-year panel

Measured at thirty seeds on forty instruments over 252 trading days. The
rows come in four groups:

- **shape**: fourteen statistics of how prices, volume and correlation
  behave, from `envelope.CERTIFIED`
- **level**: the index's yearly return, from `envelope.CERTIFIED_LEVEL`
- **crisis**: how far the VIX jumps on a day the index falls 1% or 3%, and
  how often the index falls 3% or more, from `envelope.CERTIFIED_CRISIS`
- **dispersion**: how unevenly sectors suffer when the VIX is above 30

The level and crisis rows are measured on a protocol where the roster is
drawn afresh with each seed, because a level that describes the model
cannot be read off one roster, so they sit in their own tables.

`envelope.score` grades a panel against the band real markets produce for
each row.

In [2]:
panel = {**env.certified_panel(), **env.CERTIFIED_LEVEL, **env.CERTIFIED_CRISIS}
year = env.score(panel, horizon_days=252)

print(f"ruler: {year['ruler']}\n")
print(f"{'row':26s} {'group':>10s} {'measured':>9s} {'real band':>15s}")
for name, row in year["statistics"].items():
    lo, hi = row["band"]
    print(f"{LABELS.get(name, name):26s} {row['group']:>10s} "
          f"{row['measured']:9.4f} {f'{lo:g} to {hi:g}':>15s}  "
          f"{'in' if row['in_band'] else 'OUT'}")

print(f"\n{year['in_band']} of {year['of']} in band: "
      f"shape {year['shape_in_band']}/{year['shape_of']}, "
      f"level {year['level_in_band']}/{year['level_of']}, "
      f"crisis {year['crisis_in_band']}/{year['crisis_of']}, "
      f"dispersion {year['dispersion_in_band']}/{year['dispersion_of']}")

ruler: facts.REAL_MARKETS_RULED

row                             group  measured       real band
annualised vol %                shape   20.5456        12 to 41  in
excess kurtosis                 shape   18.1072       -13 to 24  in
return acf(1)                   shape    0.0130   -0.07 to 0.06  in
|return| acf(1)                 shape    0.0282    0.02 to 0.17  in
|return| acf(5)                 shape    0.0188    -0.03 to 0.1  in
|return| acf(20)                shape    0.0044   -0.05 to 0.06  in
cross-sectional corr            shape    0.3053    0.09 to 0.49  in
volume vs |return|              shape    0.5084    0.35 to 0.64  in
leverage, r vs |r+1|            shape   -0.0341      -0.11 to 0  in
volume change acf(1)            shape   -0.2540    -0.3 to -0.2  in
corr, down vs up days           shape    0.0791   -0.15 to 0.23  in
corr, after down vs up          shape    0.0860   -0.15 to 0.33  in
same-sector excess corr         shape    0.1165    0.04 to 0.23  in
corr persistence ac

Every row is inside its band at one year.

The bands are wide on purpose. Each one is the range real markets have
produced across many one-year windows, back to 1987 for most rows and to
1928 for the index's return, so a row in band reads like some real year.
That is a weaker claim than reading like the average year.

`room_sd` says how far inside its band a row sits, in units of its own
spread across seeds. A row barely inside is one unlucky seed from
outside, and a plain in-or-out check cannot tell the two apart.

In [3]:
rows = [(k, r["room_sd"]) for k, r in year["statistics"].items()
        if r["in_band"] and r["room_sd"] is not None]
for k, room in sorted(rows, key=lambda kv: kv[1]):
    flag = "  <- thin" if room < 0.5 else ""
    print(f"  {LABELS.get(k, k):26s} {room:6.2f} sd inside{flag}")

  |return| acf(1)              0.09 sd inside  <- thin
  index drift %/yr             0.27 sd inside  <- thin
  leverage, r vs |r+1|         0.44 sd inside  <- thin
  |return| acf(5)              0.86 sd inside
  return acf(1)                0.89 sd inside
  corr, down vs up days        0.92 sd inside
  |return| acf(20)             1.16 sd inside
  annualised vol %             1.32 sd inside
  corr persistence acf(1)      1.65 sd inside
  cross-sectional corr         1.70 sd inside
  corr, after down vs up       2.04 sd inside
  volume vs |return|           3.16 sd inside
  volume change acf(1)         4.27 sd inside
  excess kurtosis              5.00 sd inside
  same-sector excess corr     11.96 sd inside


Three rows sit within half a seed standard deviation of an edge:
one-day volatility clustering (`|return| acf(1)`, 0.09 sd inside), the
index's yearly return (0.27 sd) and the leverage effect (0.44 sd). The
index drifts 7.70% a year on this protocol, inside a band of 1.1 to 10.3
(pt-v19 read 7.65), but its spread across seeds is wide. Any single run
can read those three outside their bands.

## The 2015-2025 ruler

The bands above are `envelope.score`'s default. The library also keeps an
older set cut from 2015-2025 alone, `facts.REAL_MARKETS`, which is
narrower on some rows because one decade varies less than four. Scoring
the same numbers against it shows what a user comparing the model with
the recent past would see.

In [4]:
decade = env.score(panel, horizon_days=252, basis="shipped")
print(f"{decade['ruler']}: {decade['in_band']} of {decade['of']} in band")
for name, row in decade["statistics"].items():
    if row["in_band"] is False:
        lo, hi = row["band"]
        print(f"  out: {name} {row['measured']:.4f}, band {lo:g} to {hi:g}")
print(f"  no band on this ruler: {', '.join(decade['unreadable'])}")

facts.REAL_MARKETS: 18 of 18 in band
  no band on this ruler: crisis_sector_dispersion


Every row is in on this ruler too. On pt-v19 one row was out here:
`sector_excess_corr`, how much more two stocks in the same sector move
together than two picked at random, at 0.090 against a floor of 0.11.
pt-v20 reads 0.117.

## The two-year ruler

A statistic read over two years is a different measurement from the same
statistic read over one, so it needs bands derived at two years.
`envelope.score` picks the ruler from `horizon_days`. Below is the
two-year panel, measured at 504 days, against the two-year bands.

In [5]:
two_years = {k: v for k, v in env.MEASURED_504.items() if v is not None}
for basis in ("ruled", "shipped"):
    s = env.score(two_years, horizon_days=504, basis=basis)
    out = [k for k, r in s["statistics"].items() if r["in_band"] is False]
    print(f"{s['ruler']}: {s['in_band']} of {s['of']} in band")
    print(f"    out: {', '.join(out) or 'none'};  "
          f"no band: {', '.join(s['unreadable']) or 'none'}")

facts.REAL_MARKETS_RULED_504: 14 of 14 in band
    out: none;  no band: corr_persistence_acf1
envelope.BANDS_504: 13 of 14 in band
    out: abs_return_acf1;  no band: crisis_sector_dispersion


At two years every row the long-record bands can grade is in. On the
2015-2025 bands one is out: `abs_return_acf1`, one-day volatility
clustering, reads 0.0384 against a floor of 0.04. It is the row nearest
its edge at one year as well. The envelope certifies one year only,
because that is where the thirty-seed certification was run, and `check`
below refuses a question beyond it.

## The long-run check

New in 0.8.0. The one-year panel cannot see what a long backtest would:
whether crashes are as violent as real ones, how often fear takes hold,
how many bear markets a decade brings, and what the index returns over
twenty years. The long-run check reads forty criteria on pt-v20: the
fifteen adopted on 2026-09-23, C4a and C4b added on 2026-09-24, eleven
registered for pt-v20 the same day (B9, C5 to C9, R1 to R4 and E1), D2 on
2026-09-24, F1, L1, R5, R6 and C10 on 2026-09-25, and R7a, R7b, S1a, S1b,
S2 and V1 on 2026-09-26. Each has a tolerance a
person can read, set by asking whether a gap would mislead a user rather
than by asking for an exact match.

The A rows replay 2008 and 2020 with the real VIX imposed, so they test
how the market responds to fear rather than whether it produces the fear
itself. The B rows let the market run free for twenty-one years. The C
rows look for illusions a trading bot could learn, and D1 is the one-year
panel above. C4a and C4b ask whether 65-minute returns reverse more than
real ones, and whether simple reversal and momentum rules beat
buy-and-hold on the published suite of 20 markets. C5 to C8 ask the same
of a name's own variance, a value screen, 12-1 momentum and a one-day
reversal book, and C9 what size costs in the agent-facing book. B9 reads
how much the index's annual return varies, R1 to R4 how Treasuries and
corporate bonds move against real yields and stocks, and E1 how far
aggregate earnings fall in a contraction. D2, F1 and L1 drive the real
2020-21 macro path and read the fall, its speed, the recovery, and
whether prices turn before earnings; R5 and R6 drive 2022 and read the
fall and how far the market's P/E moves per point of the corporate
yield. C10, R7a and R7b ask whether published macro data or a published
rate decision can be traded for an edge. S1a, S1b and S2 ask whether the
packaged recession recovers, and V1 whether the index's variance over two
and five years stays in proportion to one year's.

The result is part of the preset's measured record, which ships in the
wheel.

In [6]:
record = tf.preset_record(env.PRESET)
long_run = record["long_run"]
measured = long_run["measured"]
print(f"{env.PRESET}: {long_run['verdict']}, {long_run['passed']} of "
      f"{long_run['of']} criteria")
print(f"{measured['seeds']} histories of {measured['years']} years, "
      f"measured {measured['date']}\n")


def show(value):
    if isinstance(value, list):
        return " / ".join(f"{v:g}" for v in value)
    return f"{value:g}" if isinstance(value, (int, float)) else str(value)


for row in long_run["rows"]:
    print(f"{row['id']}  {'pass' if row['pass'] else 'FAIL'}  {row['words']}")
    print(f"          model {show(row['value']):>12s}   real "
          f"{show(row['real']):>12s}   {row['rule']}")

pt-v20: pass, 40 of 40 criteria
90 histories of 21 years, measured 2026-09-26

A1  pass  worst month's volatility in the 2008 and 2020 replays within 30% of real
          model  88.5 / 76.5   real  84.3 / 94.5   within 30%
A2  pass  maximum drawdown in the 2008 and 2020 replays within 30% of real
          model 0.45 / 0.368   real 0.568 / 0.339   within 30%
A3  pass  peak stock correlation in the 2008 and 2020 replays within 0.15 of real
          model 0.784 / 0.784   real 0.748 / 0.872   within 0.15
B1  pass  share of sessions with the VIX above 30
          model        0.059   real        0.082   1/2x to 2x
B2  pass  mean length of a fear spell above VIX 30, sessions
          model           27   real           22   1/2x to 2x
B3  pass  20% bear markets per decade
          model         1.96   real         1.12   1/2x to 2x
B4  pass  10% corrections per decade
          model         4.52   real         3.65   1/2x to 2x
B5  pass  sessions down more than 5% per decade
         

In plain terms, for the shipped default:

- A 2008 or 2020 replay produces a crash of roughly the right size: its
  worst month's volatility, its maximum drawdown and its peak correlation
  between stocks all land within the tolerance of the real ones.
- Over twenty free-running years the VIX spends 5.9% of sessions above 30,
  against 8.2% in real markets, and a fear spell lasts about as long, 27
  sessions against 22.
- Bear markets of 20% come 1.96 times a decade, against 1.12 in real
  markets. Corrections of 10% come 4.52 times against 3.65, and sessions
  down more than 5% come 9.6 times against 6.2.
- The index returns 6.4% a year over the long run, against a target of
  6.25%. The target is the real index's return with the long rise in
  valuations taken out: a market priced off earnings has no reason to
  reproduce a P/E ratio that drifted up for decades.
- Reading a headline five ticks late earns about 16 basis points, under
  the 20 basis point line where a bot would learn an edge that real markets
  do not offer.
- It passes C4a and C4b, which pt-v19 failed. The lag-1 autocorrelation of
  65-minute returns is -0.016, where pt-v19 read -0.187, and the price-only
  rule furthest over its line on the suite reads +0.2 points against
  buy-and-hold and beats it in 11 markets of 20. On pt-v19 a mean-reversion
  rule that trades every 65 minutes beat buy-and-hold in 18 of 20 by a
  median 13.6 points.
- A value screen on published fundamentals and 12-1 momentum rank the next
  20 sessions about as weakly as they do in real markets (C6, C7), and the
  cost of size in the book follows the square-root law, an exponent of
  0.48 against 0.5 (C9).
- Driven by the real 2020-21 macro path, the index falls 37.4% against the
  S&P 500's 33.9% and is back at its high in 120.5 sessions against 126
  (D2). A timing rule on published macro data gains at most 0.12 points a
  year over buy-and-hold (C10), and trading on a published rate decision
  loses 0.32 points a year (R7b).

A pass is not always a wide one. The next cell measures how much of each
criterion's tolerance the model uses, for the criteria whose rule it can
read: 0 means it matches real markets exactly, 1 means it sits on the
edge.

In [7]:
def used(row):
    rule = row["rule"]
    pairs = (list(zip(row["value"], row["real"]))
             if isinstance(row["value"], list) else [(row["value"], row["real"])])
    out = []
    for model, real in pairs:
        if not isinstance(model, (int, float)):
            return None
        if rule == "within 30%":
            out.append(abs(model / real - 1) / 0.30)
        elif rule == "within 20%":
            out.append(abs(model / real - 1) / 0.20)
        elif rule == "within 0.15":
            out.append(abs(model - real) / 0.15)
        elif rule.startswith("within 2 points"):
            out.append(abs(model - real) / 2)
        elif rule == "1/2x to 2x":
            out.append(abs(math.log(model / real)) / math.log(2))
        elif rule == "2/3x to 1.5x":
            out.append(math.log(model / real)
                       / math.log(1.5 if model >= real else 2 / 3))
        elif rule == "at most 1":
            out.append(model / 1)
        elif rule == "under 20 bp":
            out.append(model / 20)
        else:
            return None
    return max(out)


scored = [(row["id"], used(row), row["words"]) for row in long_run["rows"]]
for rid, share, words in sorted(scored, key=lambda t: -(t[1] or 0)):
    if share is None:
        continue
    flag = "  <- near the edge" if share >= 0.85 else ""
    print(f"  {rid}  {share:4.0%}  {words[:58]}{flag}")

  B3   81%  20% bear markets per decade
  C3   79%  edge from reading a headline 5 ticks late, bp
  A2   69%  maximum drawdown in the 2008 and 2020 replays within 30% o
  A1   63%  worst month's volatility in the 2008 and 2020 replays with
  B5   63%  sessions down more than 5% per decade
  A3   59%  peak stock correlation in the 2008 and 2020 replays within
  B1   47%  share of sessions with the VIX above 30
  B4   31%  10% corrections per decade
  B2   30%  mean length of a fear spell above VIX 30, sessions
  B6   30%  share of sessions with the VIX under 15
  B7   28%  index annual volatility, %
  C1   21%  crash rate in years 3-21 against years 1-2
  B8    8%  long-run index return, % a year
  C2    0%  histories touching the VIX ceiling, of 30


Of the criteria the cell can read, the one nearest its edge is bear
markets per decade: 1.96 against 1.12, 1.75 times as often, using 81% of
the tolerance. The grade reports four rows nearer the edge than that among
the others: C10's worst timing rule at 92%, L1 and R1 at 86% (prices turn
10.5 sessions before earnings, against 68 in 2020; the 2-year Treasury yield
moves 3.87 bp a day against 5.23), and S1a at 84% (the recession wins
back 49% of its fall in a year, against 62% in 2009). The 2020 replay's
worst month has a volatility of 76.5 against a real 94.5, about 19% milder
(pt-v19 67, 30% milder), and its peak correlation between stocks is 0.78
against 0.87.

So a 2020 replay is recognisably a crash, and its worst stretch is gentler
than the real one. A strategy that survives it here has survived something
smaller than March 2020, and a long backtest here meets bear markets more
often than history would give it.

## The gaps that remain

Three measurements sit outside the pass bar by the owner's ruling: they
are reported and investigated, and they do not stop a preset. They are on
the record below.

In [8]:
lever = record["crisis_lever"]
print(f"crisis lever: volatility {lever['vol_at_vix_5']:.1f}% with the VIX "
      f"held at 5, {lever['vol_at_vix_65']:.1f}% held at 65")
print(f"              {lever['ratio']:.2f}x, against {lever['real']:.2f}x "
      f"in real markets\n")

rise = record["structure_rise"]["rows"]["vix_ar1_debiased"]
for days in (252, 504):
    print(f"VIX persistence over {days} days: {rise[f'median_{days}']:.3f}, "
          f"real {facts.REAL_VIX_AR1[days]:.3f}")

crisis lever: volatility 16.5% with the VIX held at 5, 84.6% held at 65
              5.11x, against 6.16x in real markets

VIX persistence over 252 days: 0.930, real 0.930
VIX persistence over 504 days: 0.941, real 0.959


- **The crisis lever is short of real.** Holding the VIX at 65 instead of
  5 makes stocks 5.1 times as volatile here, against 6.2 times in real
  markets. A scenario that forces the VIX high moves prices less than a
  real panic at the same VIX would. `pt-v19`, the previous default, read
  5.22x on the same measurement, and `pt-v18` 7.06x, which overshot.
- **Fear fades a little too fast over two years.** Over one year the VIX's
  day-to-day persistence matches real markets, 0.930 against 0.930. Over
  two it reads 0.941 against 0.959, so a long fear spell unwinds a little
  sooner than a real one.
- **Volatility memory is short.** How strongly a volatile day predicts
  volatility weeks later decays too quickly. The `decay-shape` gap below
  gives the numbers.

The named gaps each say what they stop you concluding.

In [9]:
for gap in env.GAPS:
    print(f"* {gap.id}: {gap.summary}")
    print(f"    forbids: {gap.forbids}")

* horizon: the certified horizon is 252 days
    forbids: multi-year backtests, and anything keyed on volatility dynamics beyond one year
* decay-shape: volatility memory is weaker than real at every lag
    forbids: strategies whose edge depends on volatility memory beyond about lag 20, such as vol targeting and risk parity on a one-month or longer estimate
* scenario-magnitude: a driven scenario moves prices at a quarter to a half of the real size
    forbids: sizing a scenario's impact rather than detecting it
* macro-range: the endogenous macro state cannot reach its own crisis regimes
    forbids: studying inflation regimes or policy crises from the endogenous economy alone
* roster-concentration: a concentrated roster is measured on pt-v19 only, for four sector mixes and the shape rows
    forbids: citing the certification for a concentrated roster on a level or crisis row, past 504 days, on any preset but pt-v19 (the default pt-v20 included), or for a sector mix other than the f

## Checking your own question

`envelope.check` takes a horizon, the statistics a strategy relies on, and
the shape of the roster, and says whether the question is inside the
envelope. Every refusal names the measurement behind it.

In [10]:
questions = [
    ("a one-year momentum study", dict(horizon_days=252,
                                       statistics=["return_acf1"])),
    ("a three-year study",        dict(horizon_days=756,
                                       statistics=["abs_return_acf1"])),
    ("volatility clustering decay", dict(horizon_days=252,
                                         statistics=["abs_return_acf20"])),
    ("a tech-only roster",        dict(horizon_days=252,
                                       sector_concentrated=True)),
    ("sizing a crisis scenario",  dict(horizon_days=252,
                                       scenario_magnitude=True)),
]

import textwrap

for label, kwargs in questions:
    v = env.check(**kwargs)
    print(f"{label:32s} {'INSIDE' if v.inside else 'outside'}")
    if not v.inside:
        for reason in v.reasons:
            print(textwrap.fill(textwrap.shorten(reason, 240), 76,
                                initial_indent="      ",
                                subsequent_indent="      "))

a one-year momentum study        INSIDE
a three-year study               outside
      horizon 756d exceeds the certified 252d. At 504 days the model holds
      all 13 shape rows the ruled bands can grade
      (facts.REAL_MARKETS_RULED_504); corr_persistence_acf1 has no band
      there. On the 2015-2025 decade bands (BANDS_504) it [...]
volatility clustering decay      outside
      abs_return_acf20 depends on the decay shape, which is a mechanism gap:
      the |return| autocorrelation reads below real markets' at every lag
      (0.0282 against 0.1071 at lag 1, 0.0044 against 0.0286 at lag 20), is
      resolved as positive only [...]
a tech-only roster               outside
      the roster is sector-concentrated and its mix is not named. Four mixes
      are measured on pt-v19 over 30 seeds at 252 and 504 days on the ruled
      bands (tools/calibration/roster_shapes.py, fleet run docs080b,
      2026-09-24), and each held [...]
sizing a crisis scenario         outside
      the

The three-year refusal grades the two-year panel on both rulers, as the
cell above did: every row in on the long-record bands, and
`abs_return_acf1` out on the 2015-2025 ones. It refuses on the horizon.

`check` reads the one-year envelope, so it refuses a three-year study even
though the long-run check passes all forty of its criteria. The two
answer different questions. The
long-run check says that crash sizes, fear, bear markets and long-run
return look like a real market's over twenty years. The envelope says
which detailed statistics are certified, and it certifies them at one year
and no further. A multi-year backtest is on firmer ground than it was
before 0.8.0. It is still outside what `check` certifies, so test it
across seeds rather than trusting one run.

## Choosing a preset

`pt-v20` is the default from 0.8.5; `pt-v19` was the default from 0.8.0
and `pt-v18` through 0.7.x. Every earlier preset stays selectable, so work published against
one of them keeps reproducing. Each shipped default carries a measured
record, and `envelope.regressions` names the one-year rows a preset gives
up against the shipped one.

In [11]:
defaults = ("pt-v3", "pt-v10", "pt-v12", "pt-v14", "pt-v16", "pt-v18", "pt-v19",
            "pt-v20")
print(f"{'preset':8s} {'default':>8s}  {'crisis lever':>12s}  {'long run':>9s}  "
      f"gives up against {env.PRESET} at one year")
for name in defaults:
    r = tf.preset_record(name)
    lost = env.regressions({k: v for k, v in r["panel_252"].items()
                            if v is not None})
    lr = r.get("long_run")
    verdict = f"{lr['passed']}/{lr['of']}" if lr else "not run"
    print(f"{name:8s} {r['default_since']:>8s}  {r['crisis_lever']['ratio']:11.2f}x"
          f"  {verdict:>9s}  {', '.join(lost) or 'nothing'}")

preset    default  crisis lever   long run  gives up against pt-v20 at one year
pt-v3       0.1.0         3.08x    not run  sector_excess_corr, volume_change_acf1
pt-v10      0.2.0         5.04x    not run  volume_change_acf1
pt-v12      0.3.0         5.95x    not run  nothing
pt-v14      0.4.0         6.19x    not run  nothing
pt-v16      0.6.0         6.50x    not run  nothing
pt-v18      0.7.0         7.06x    not run  nothing
pt-v19      0.8.0         5.22x      15/17  nothing
pt-v20      0.8.5         5.11x      40/40  nothing


On the one-year panel the defaults from `pt-v12` on are hard to tell
apart: none gives up a row. `pt-v3` gives up sector co-movement and
volume-change autocorrelation, and `pt-v10` the second of those. What
separates the last two is the long-run check, which is on record for them
alone. `pt-v20` passes all forty criteria. `pt-v19`'s record carries the
seventeen it was graded on and fails two, the price-only rows; graded on
all forty beside pt-v20, it fails sixteen. Their crisis levers are close,
5.11x against 5.22x, both below real.

**Take the default, `pt-v20`, unless you are reproducing work published
against an older preset. For a study longer than a year, read the long-run
check above, run several seeds, and remember that `check` does not
certify it.**

## Summary

- Realism is a set of measurements against real-market bands, with the
  failures named, and no single score.
- At one year every row of the panel is in band on both rulers. At two
  years one row, one-day volatility clustering, is just under the
  2015-2025 floor.
- `pt-v20` passes all forty long-run criteria, including the two
  price-only rows `pt-v19` fails. The nearest the edge are a timing rule
  on published macro data, the lead of prices over earnings in 2020, the
  2-year Treasury yield's daily moves and the recession's recovery, and a
  2020 replay's worst month is about 19% milder than the real one.
- The crisis lever, two-year VIX persistence and volatility memory are
  still off, and are on the record.
- Good results here do not predict real returns.

Full documentation: <https://tradefloor.dev>